# Platform Narrative

This notebook is the canonical architecture story for the ML deployment reference.

## Executive summary

We are building a **notebook-first ML platform reference** with:

- **MLflow** as the system of record for experiments, runs, models, and lineage
- **PostgreSQL + S3** as the MLflow persistence baseline (MinIO locally for parity)
- **Lambda.ai + Slurm** for distributed training and inference coordination
- **AWS + Kubernetes** for non-Lambda platform services and production operations
- **python-terraform** for Python-managed infrastructure workflows
- **Web UI + immutable notebook revisions** for controlled execution without notebook editing

## Platform architecture

```{mermaid}
flowchart LR
    U["ML Engineer"] --> UI["Notebook Web UI (immutable revision selection)"]
    UI --> ORCH["Execution Orchestrator (contract and policy checks)"]

    subgraph LOCAL["Local parity environment"]
      LOC_RUN["Local runner"]
      LOC_PG["PostgreSQL"]
      LOC_S3["MinIO or S3-compatible storage"]
      LOC_K8S["Kubernetes-like local control plane"]
      LOC_SLRM["Slurm-like local scheduling path"]
    end

    subgraph LAMBDA["Lambda.ai distributed compute"]
      SLRM["Slurm controller"]
      TRAIN["Distributed training and inference jobs"]
    end

    subgraph AWS["AWS platform services"]
      K8S["EKS or Kubernetes services"]
      S3["S3 artifacts"]
      RDS["RDS PostgreSQL"]
    end

    ORCH --> LOC_RUN
    ORCH --> SLRM
    ORCH --> K8S

    LOC_RUN --> MLFLOW["MLflow tracking and registry"]
    TRAIN --> MLFLOW
    K8S --> MLFLOW

    MLFLOW --- LOC_PG
    MLFLOW --- LOC_S3
    MLFLOW --- RDS
    MLFLOW --- S3

    LOC_K8S -."local parity".-> K8S
    LOC_SLRM -."local parity".-> SLRM

    MLFLOW --> OBS["Observability and cost visibility"]
    OBS --> U
```


## Reading lenses

### Lifecycle lens
Data and lineage -> training and experimentation -> model registry and promotion -> serving and observability.

### Topology lens
Local parity -> Lambda.ai Slurm distributed compute -> AWS Kubernetes platform services.

### Operations lens
Terraform-driven environment provisioning, security and governance controls, and cost attribution.

## Non-negotiable constraints represented

1. Python-first, Linux-only, PyTorch + GPU posture
2. Docker-first reproducibility, with Nix as helper layer
3. MLflow with PostgreSQL backend and S3 artifact storage
4. Lambda.ai Slurm for distributed coordination and failure handling
5. AWS Kubernetes for non-Lambda platform services
6. python-terraform as default infrastructure workflow
7. Security, lineage, experiment/model traceability, and reproducibility as hard requirements